# Visualization

In [ ]:
# IMPORTS ###########################################

import importlib
import warnings
warnings.filterwarnings("ignore")

#####################################################

import numpy as np
from scipy import signal

from pxg import plot
importlib.reload(plot)

from pxg import Stop
from pxg import FS, MS
from pxg import EXG, Rids, Record

from IPython.display import Markdown, display

%config InlineBackend.figure_format = "retina"

In [ ]:
# Visualization

def PlotPage(
    rec: Record, 
    page = 0, 
    offset = 0,
    marker = False,
    include = []
):
    res = plot.Page(
        rec, 
        page, 
        offset,

        SENSOR = False,
        SIGNAL = True,
        DECTOR = False,
        EXTEND = None,

        # LOW = -700,
        ZEROS = [0],

        # label = "BPM",
        angle = 0,

        rrqs  = False, 
        qsvl  = False, 
        
        punts = False,  
        trig  = False,
        onoff = False,
        letra = False,

        grid  = False,
        anref = False,
        simple = True,

        # simple = False,
        # marker = marker and page not in include,

        show = False,
    )

    if res:
        PON, POF = plot.Range(rec, page, offset)

        # plot.Signal(rec.Local[PON:POF] / MV, zero=-400, color="tab:blue", format="-", linewidth=0.6)
        # plot.Signal(rec.Cover[PON:POF] / MV, zero=-400, color="tab:orange", format="-", linewidth=0.5)

        # plot.Signal(rec.Maxim[PON:POF] / MV, zero=-400, color="tab:red", format="-", linewidth=0.8)
        # plot.Signal(rec.Minim[PON:POF] / MV, zero=-400, color="tab:red", format="-", linewidth=0.8)

        plot.Signal(rec.Digit[PON:POF] * 90, zero=-700, color="tab:gray", format="-", linewidth=0.5, fill=True, alpha=0.2)
        plot.Signal(rec.Class[PON:POF] * 90, zero=-700, color="tab:red", format="-", linewidth=0.5, fill=True, alpha=0.2)
    pass #if

    plot.Show()

    return res
pass #def


def PlotRecord(rec: Record, page: int | list[int] = -1, off = 0, marker = False, title = ""):
    offset = off*plot.FS
    display(Markdown(f"## {rec.DB.upper()} {rec.RID}\n---"))
    if title:
        display(Markdown(f"```text\n{title}\n```"))
    pass #if
    if isinstance(page, list):
        for p in page:
            PlotPage(rec, p, offset, marker = marker)
        pass #for
    elif page != -1:
        PlotPage(rec, page, offset, marker = marker)
    else:
        for page in range(0, (len(rec.Signal) + plot.CHUNK // 2) // plot.CHUNK, 1):
            PlotPage(rec, page, offset, marker = marker)
        pass #for
    pass #if
    return rec
pass #def

def PlotDatabase(db: str):
    rids = EXG(db, learn = True, cfm = True, devx = False, wrx = False)
    # rids = Rids(db)
    for rid in rids:
        rec = Record(db, rid)
        for page in range(0, (len(rec.Signal) + plot.CHUNK // 2) // plot.CHUNK, 1):
            PON, POF = plot.Range(rec, page, 0)
            cc = 0
            for epi in rec.RefEpi:
                if epi.End < PON or epi.Time >= POF: continue
                if epi.Name in ["VT", "VFL", "VF", "WF"]: cc+=1
            pass #for
            if cc == 0: continue
            PlotRecord(rec, page = [page], off = 0, marker = False, title = "")
        pass #for
    pass #for
pass #def


In [ ]:
PlotDatabase("mitdb")
PlotDatabase("cudb")